# 기프트코 상품 추천 RAG v0 Starter

이 노트북은 **상품데이터와 거래데이터가 1:1로 매칭되지 않는 상황**을 전제로 만든 상품 추천 v0입니다.

## 핵심 방향

```text
상품데이터 = 현재 추천 가능한 상품 후보 DB
거래데이터 = 과거 구매 패턴/업종/행사 힌트 DB
```

따라서 두 파일을 억지로 상품명 매칭하지 않습니다.

## v0 추천 구조

```text
사용자 질문
↓
예산/수량 조건 추출
↓
거래데이터에서 비슷한 과거 사례 검색
↓
과거 사례의 상품분류 힌트 추출
↓
상품데이터에서 가격/수량 조건 반영
↓
상품명/카테고리/키워드 기반 검색
↓
Top 5 추천
```

> 이 v0는 완성형 추천기가 아니라, 상품 추천 RAG 구조를 검증하기 위한 실험용 노트북입니다.


## 0. 패키지 설치

처음 한 번만 실행하면 됩니다.


In [1]:
# 필요 시 한 번만 실행
# !pip install pandas openpyxl scikit-learn


## 1. 기본 설정 및 파일 경로

샘플 파일 기준으로 설정되어 있습니다.

원본 대용량 파일로 바꾸려면 `PRODUCT_PATH`, `TRADE_PATH`만 수정하면 됩니다.


In [2]:
from pathlib import Path
import re
import html
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

# =========================
# 파일 경로
# =========================
PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

# ChatGPT 샌드박스 경로에서 실행하는 경우
if not PRODUCT_PATH.exists():
    PRODUCT_PATH = Path("/mnt/data/상품데이터_샘플100.xlsx")
if not TRADE_PATH.exists():
    TRADE_PATH = Path("/mnt/data/거래데이터_샘플100.xlsx")

OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("PRODUCT_PATH:", PRODUCT_PATH)
print("TRADE_PATH:", TRADE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

PRODUCT_PATH: ..\data\상품데이터_샘플100.xlsx
TRADE_PATH: ..\data\거래데이터_샘플100.xlsx
OUTPUT_DIR: C:\Users\USER\Desktop\workspace\work-code-archive-organized\projects\rag-experiments\product-recommendation\notebooks\output


## 2. 엑셀 로딩 및 컬럼 확인

현재 샘플 기준:

- 거래데이터: 100행 × 23컬럼
- 상품데이터: 100행 × 93컬럼

컬럼명이 다르면 아래 컬럼 매핑 부분에서 수정하면 됩니다.


In [3]:
# =========================
# 엑셀 로딩
# =========================

product_raw = pd.read_excel(PRODUCT_PATH, dtype=str).fillna("")
trade_raw = pd.read_excel(TRADE_PATH, dtype=str).fillna("")

print("상품데이터 shape:", product_raw.shape)
print("거래데이터 shape:", trade_raw.shape)

print("\n[상품데이터 컬럼]")
for i, col in enumerate(product_raw.columns, start=1):
    print(f"{i:02d}. {col}")

print("\n[거래데이터 컬럼]")
for i, col in enumerate(trade_raw.columns, start=1):
    print(f"{i:02d}. {col}")

display(product_raw.head(3))
display(trade_raw.head(3))


상품데이터 shape: (100, 93)
거래데이터 shape: (100, 23)

[상품데이터 컬럼]
01. 일련번호
02. 상품번호
03. 입점업체ID
04. 브랜드
05. 상품명
06. 간략한설명
07. 모델명
08. 상품코드(바코드)
09. 상품판매가
10. 상품공급가
11. 수수료율
12. 시중가격
13. 네이버최저가설정
14. 적립금
15. 재고
16. 옵션명1
17. 옵션아이템1
18. 옵션1재고
19. 옵션1가격
20. 옵션명2
21. 옵션아이템2
22. 옵션명3
23. 옵션아이템3
24. 옵션명4
25. 옵션아이템4
26. 옵션명5
27. 옵션아이템5
28. 검색키워드
29. 대 카테고리
30. 중 카테고리
31. 소 카테고리
32. 세분류
33. 대 카테고리(중복)
34. 중 카테고리(중복)
35. 소 카테고리(중복)
36. 세분류(중복)
37. 큰이미지
38. 중간이미지
39. 작은이미지
40. 다른이미지
41. 상품내용
42. 과세/면세
43. 제조사
44. 원산지
45. 상품크기
46. 용량
47. 용량단위
48. 제목1
49. 내용1
50. 이벤트문구
51. 배송비구분
52. 배송비
53. 배송비무료금액
54. 배송정책
55. 최소구매수량설정
56. 최소구매수량
57. 구매단위
58. 복수구매
59. 복수구매수량계산방식
60. 첨부파일
61. 회원전용상품
62. 회원전용상품회원등급
63. 정보고시등록여부
64. 정보고시카테고리
65. 정보고시_1 내용
66. 정보고시_2 내용
67. 정보고시_3 내용
68. 정보고시_4 내용
69. 정보고시_5 내용
70. 정보고시_6 내용
71. 정보고시_7 내용
72. 정보고시_8 내용
73. 정보고시_9 내용
74. 정보고시_10 내용
75. 정보고시_11 내용
76. 정보고시_12 내용
77. 정보고시_13 내용
78. 정보고시_14 내용
79. 정보고시_15 내용
80. Unnamed: 79
81. Unnamed: 80
82. Unnamed: 81
83. 상품코드
84. 폐쇄몰공급가
85. 폐

,일련번호,상품번호,입점업체ID,브랜드,상품명,간략한설명,모델명,상품코드(바코드),상품판매가,상품공급가,수수료율,시중가격,네이버최저가설정,적립금,재고,옵션명1,옵션아이템1,옵션1재고,옵션1가격,옵션명2,옵션아이템2,옵션명3,옵션아이템3,옵션명4,옵션아이템4,옵션명5,옵션아이템5,검색키워드,대 카테고리,중 카테고리,소 카테고리,세분류,대 카테고리(중복),중 카테고리(중복),소 카테고리(중복),세분류(중복),큰이미지,중간이미지,작은이미지,다른이미지,...,배송정책,최소구매수량설정,최소구매수량,구매단위,복수구매,복수구매수량계산방식,첨부파일,회원전용상품,회원전용상품회원등급,정보고시등록여부,정보고시카테고리,정보고시_1 내용,정보고시_2 내용,정보고시_3 내용,정보고시_4 내용,정보고시_5 내용,정보고시_6 내용,정보고시_7 내용,정보고시_8 내용,정보고시_9 내용,정보고시_10 내용,정보고시_11 내용,정보고시_12 내용,정보고시_13 내용,정보고시_14 내용,정보고시_15 내용,Unnamed: 79,Unnamed: 80,Unnamed: 81,상품코드,폐쇄몰공급가,폐쇄몰판매가,해외판매공급가,리워드판매수익퍼센트,리워드소개수익퍼센트,Unnamed: 88,Unnamed: 89,Unnamed: 90,Unnamed: 91,Unnamed: 92
0,23271,36601,osiris0000,,[PQI] USB 메모리 U273V USB3.0 8GB~,,,,10925,9500,60,0,,0,,,,,,,,,,,,,,"pqi,usb,메모리,usb메모리,외장하드,외장디스크,pqiusb",가전/디지털/USB,USB메모리,캡방식USB,,,,,,08/14750008.jpg,08/14750008.jpg,08/14750008.jpg,,...,,설정함,1000,1,"10:13300,30:12920,50:12540,100:11780,300:11495,500:11210,1000:10925",개별,,,,무,,,,,,,,,,,,,,,,,,,,14750008,0,0,0,0,0,,,,,
1,11571,79362,thegreen,,오버핏 라운드 티셔츠,,,,6435,5500,0,0,,0,,,,,,,,,,,,,,#티셔츠#라운드#레이어드,단체복,티셔츠,가방/의류,,,,,,오버핏-라운드-티셔츠(17수)-썸네일(500)-수정.jpg,오버핏-라운드-티셔츠(17수)-썸네일(500)-수정.jpg,오버핏-라운드-티셔츠(17수)-썸네일(500)-수정.jpg,,...,,설정함,5000,1,"50:7205,100:6985,300:6875,500:6765,1000:6655,3000:6545,5000:6435",전체,,,,무,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,22636,39442,hyuga11,,오로라노크 볼펜,,,,175,155,60,0,,0,,,,,,,,,,,,,,,볼펜/사무/문구,볼펜/필기류,100원~500원,,,,,,10/15440210.jpg,10/15440210.jpg,10/15440210.jpg,,...,,설정함,30000,1,"500:214,1000:205,3000:195,5000:189,10000:181,20000:177,30000:175",개별,,,,무,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


,번호,구매처 분류(대),구매처 분류(중),구매처 분류(소),구매처 분류(세),구매처 명,날짜,상품,상품분류(대),상품분류(중),상품분류(소),대량가격(원),중간가격(원),소량가격(원),최소구매수량,행사별(필터),대상별(필터),시즌별(필터),인쇄 방법,비 고,_출처폴더,_출처파일,구매처 명
0,702,기업/회사,제약/의료기기/바이오,의료기기/의료용품,의료기기/의료용품,파라곤CRT,2023/09/25,쏘쏘 원터치 스텐텀블러 500ml 1P,일상용품,주방용품,저장용기,4500,4578,5374,20,,,,칼라전사필름인쇄,,기업회사(16692),판촉물_거래_데이터_조사_자료_기업회사_제약의료기기바이오_完(973).xlsx,
1,,기업/회사,일반기업/회사,화장품/패션/주얼리,화장품,Changefit,2021/12/14,귀요미 캐릭터 스마트링 1P,디지털/가전,휴대폰액세서리,기타휴대폰액세서리,850,873,989,50,,,,칼라인쇄,,기업회사(16692),판촉물_거래_데이터_조사_자료_기업회사_일반기업회사_完(4909).xlsx,
2,,기업/회사,일반기업/회사,외국계회사/글로벌기업,외국계회사/글로벌기업,CONMAD,2022/06/27,에스모도 미니 핸디형 선풍기 159,디지털/가전,건강/냉난방가전,선풍기,7240,7556,8458,10,,,,칼라인쇄,,기업회사(16692),판촉물_거래_데이터_조사_자료_기업회사_일반기업회사_完(4909).xlsx,


## 3. 대용량 원본 샘플링 코드

나중에 원본 파일이 너무 클 때 사용하는 샘플링 코드입니다.

단순 랜덤 샘플만 뽑으면 중요한 카테고리가 빠질 수 있으므로,

```text
랜덤 샘플
+
카테고리별 균등 샘플
```

을 섞는 방식입니다.


In [4]:
# =========================
# 대용량 원본 샘플링 함수
# =========================

def make_sample_excel(
    input_path,
    output_path,
    n_random=300,
    group_cols=None,
    n_per_group=5,
    random_state=42,
):
    """
    대용량 엑셀에서 샘플 파일을 생성합니다.

    input_path: 원본 엑셀 경로
    output_path: 저장할 샘플 엑셀 경로
    n_random: 전체 랜덤 샘플 수
    group_cols: 카테고리 균등 샘플 기준 컬럼 리스트
    n_per_group: 그룹별 샘플 수
    """

    df = pd.read_excel(input_path, dtype=str).fillna("")
    print("원본 행 수:", len(df))
    print("원본 컬럼 수:", len(df.columns))

    samples = []

    # 1) 전체 랜덤 샘플
    if len(df) > 0:
        random_sample = df.sample(
            n=min(n_random, len(df)),
            random_state=random_state,
        )
        samples.append(random_sample)

    # 2) 그룹별 균등 샘플
    if group_cols:
        valid_group_cols = [c for c in group_cols if c in df.columns]
        if valid_group_cols:
            grouped_sample = (
                df.groupby(valid_group_cols, dropna=False, group_keys=False)
                .apply(lambda x: x.sample(n=min(n_per_group, len(x)), random_state=random_state))
            )
            samples.append(grouped_sample)
        else:
            print("주의: group_cols에 지정한 컬럼이 파일에 없습니다:", group_cols)

    if samples:
        sample_df = pd.concat(samples, ignore_index=True).drop_duplicates()
    else:
        sample_df = df.head(n_random).copy()

    sample_df.to_excel(output_path, index=False)
    print("샘플 저장 완료:", output_path)
    print("샘플 행 수:", len(sample_df))

    return sample_df


# 사용 예시 1: 상품데이터 샘플링
# sample_product = make_sample_excel(
#     input_path="상품데이터_원본.xlsx",
#     output_path="상품데이터_샘플.xlsx",
#     n_random=300,
#     group_cols=["대 카테고리", "중 카테고리"],
#     n_per_group=5,
# )

# 사용 예시 2: 거래데이터 샘플링
# sample_trade = make_sample_excel(
#     input_path="거래데이터_원본.xlsx",
#     output_path="거래데이터_샘플.xlsx",
#     n_random=300,
#     group_cols=["구매처 분류(대)", "구매처 분류(중)", "상품분류(중)"],
#     n_per_group=5,
# )


## 4. 전처리 유틸 함수

상품명, 카테고리, 가격, 최소구매수량 등을 정리합니다.


In [5]:
# =========================
# 전처리 유틸
# =========================

def clean_text(x):
    """HTML 태그, 과도한 공백 등을 제거합니다."""
    if pd.isna(x):
        return ""

    x = str(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def to_number(x):
    """가격/수량 문자열을 숫자로 변환합니다."""
    if pd.isna(x):
        return None

    s = str(x).strip()

    if s in ["", "-", "nan", "None", "null"]:
        return None

    s = re.sub(r"[^0-9.]", "", s)

    if s == "":
        return None

    try:
        return float(s)
    except Exception:
        return None


def safe_col(df, col):
    """컬럼이 없을 때도 오류 없이 빈 문자열 Series를 반환합니다."""
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def combine_text(*parts):
    return clean_text(" ".join([str(p) for p in parts if str(p).strip()]))


## 5. 상품데이터 정규화

샘플 상품데이터에서 추천에 필요한 컬럼만 정리합니다.

주요 사용 컬럼:

- 상품번호
- 상품코드
- 상품명
- 상품판매가
- 최소구매수량
- 검색키워드
- 대 카테고리 / 중 카테고리 / 소 카테고리
- 간략한설명
- 상품내용


In [6]:
# =========================
# 상품데이터 정규화
# =========================

product_df = product_raw.copy()

product_df["product_id"] = safe_col(product_df, "상품번호")
product_df.loc[product_df["product_id"].str.strip() == "", "product_id"] = safe_col(product_df, "상품코드")

product_df["product_name"] = safe_col(product_df, "상품명").map(clean_text)
product_df["price"] = safe_col(product_df, "상품판매가").map(to_number)
product_df["supply_price"] = safe_col(product_df, "상품공급가").map(to_number)
product_df["moq"] = safe_col(product_df, "최소구매수량").map(to_number)

product_df["category_large"] = safe_col(product_df, "대 카테고리").map(clean_text)
product_df["category_middle"] = safe_col(product_df, "중 카테고리").map(clean_text)
product_df["category_small"] = safe_col(product_df, "소 카테고리").map(clean_text)

product_df["category_path"] = (
    product_df["category_large"] + " > " +
    product_df["category_middle"] + " > " +
    product_df["category_small"]
).map(clean_text)

product_df["keywords"] = safe_col(product_df, "검색키워드").map(clean_text)
product_df["summary"] = safe_col(product_df, "간략한설명").map(clean_text)
product_df["detail_text"] = safe_col(product_df, "상품내용").map(clean_text)
product_df["image"] = safe_col(product_df, "큰이미지").map(clean_text)

product_df["search_text"] = (
    product_df["product_name"] + " " +
    product_df["category_path"] + " " +
    product_df["keywords"] + " " +
    product_df["summary"] + " " +
    product_df["detail_text"]
).map(clean_text)

# 상품명이 없는 행 제거
product_df = product_df[product_df["product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 상품 수:", len(product_df))
display(product_df[[
    "product_id", "product_name", "price", "moq",
    "category_path", "keywords"
]].head(10))


정규화 상품 수: 100


,product_id,product_name,price,moq,category_path,keywords
0,36601,[PQI] USB 메모리 U273V USB3.0 8GB~,10925.0,1000.0,가전/디지털/USB > USB메모리 > 캡방식USB,"pqi,usb,메모리,usb메모리,외장하드,외장디스크,pqiusb"
1,79362,오버핏 라운드 티셔츠,6435.0,5000.0,단체복 > 티셔츠 > 가방/의류,#티셔츠#라운드#레이어드
2,39442,오로라노크 볼펜,175.0,30000.0,볼펜/사무/문구 > 볼펜/필기류 > 100원~500원,
3,87400,[ALIO] 모던클락 15W 우드형 LED시계&고속무선충전기,12430.0,3000.0,스마트폰용품 > 휴대폰 무선충전패드 > 가전/디지털/USB,#led시계 #시계 #우드시계 #나무시계 #디지털시계 #고속무선충전패드 #충전패드 #무선충전 #무선충전패드 #무선충전시계 #시계패드추천 #무선패드 #고속무선시계
4,101669,[유시몰] 유덴트세트 6호(치약(20g)+알칫솔+가글(15ml)+크리넥스쿨링물티슈),4163.0,5000.0,가정/생활용품 > 치약칫솔세트외 > 치약/칫솔세트,#선물세트 #유시몰 #크리넥스 #칫솔치약세트 #물티슈
5,31300,패밀리여행용세트 대1,23595.0,3000.0,레저/건강/차량 > 여행용품 > 여행용세면도구,#여행세트#여행용세트#세면세트
6,83327,국산 스페이스 포켓 시장가방,2825.0,10000.0,에코백/가방 > 장바구니 > 가방/의류,"#국산가방, #시장가방, #장바구니"
7,85324,[크로커다일] 3단 65-10K 우드 곡자(자동) 우산,16215.0,1000.0,가정/생활용품 > 우산/우의 > 3단우산,#우산 #3단우산#자동우산#크로커다일
8,3814,메탈레드 30매 물티슈,263.0,100000.0,가정/생활용품 > 물티슈/휴지 > 30매 이상 물티슈,"물휴지,물티슈,티슈,전도용물티슈,홍보용물티슈,휴대용물티슈,광고,판촉,전도티슈,물티슈,홍보용티슈,광고용물티슈,광고물티슈,홍보용,판촉물,수미향"
9,92474,[참그린] 매실주방세제펌프 피죤 베이킹소다300리필 3종세트,4344.0,3000.0,주방/식품/텀블러 > 주방/세탁 세제 > 주방/세탁 혼합세트,"주방세제, 1종 주방세제, 친환경 주방세제, 무료배송,세제선물세트,펌프주방세제,설거지, 세제,매실,퐁퐁, 식기세척, 트리오"


## 6. 거래데이터 정규화

거래데이터는 직접 추천 상품으로 쓰지 않고, **과거 구매 패턴 힌트**로 사용합니다.

주요 사용 컬럼:

- 구매처 분류(대/중/소/세)
- 구매처 명
- 상품
- 상품분류(대/중/소)
- 대량가격/중간가격/소량가격
- 최소구매수량
- 인쇄 방법


In [7]:
# =========================
# 거래데이터 정규화
# =========================

trade_df = trade_raw.copy()

trade_df["trade_product_name"] = safe_col(trade_df, "상품").map(clean_text)

trade_df["buyer_type"] = (
    safe_col(trade_df, "구매처 분류(대)") + " > " +
    safe_col(trade_df, "구매처 분류(중)") + " > " +
    safe_col(trade_df, "구매처 분류(소)") + " > " +
    safe_col(trade_df, "구매처 분류(세)")
).map(clean_text)

# 거래데이터에는 같은 이름의 '구매처 명' 컬럼이 중복될 수 있어 실제 있는 컬럼을 우선 사용
buyer_name_col = "구매처 명 "
if buyer_name_col not in trade_df.columns and "구매처 명" in trade_df.columns:
    buyer_name_col = "구매처 명"

trade_df["buyer_name"] = safe_col(trade_df, buyer_name_col).map(clean_text)

trade_df["trade_category_large"] = safe_col(trade_df, "상품분류(대)").map(clean_text)
trade_df["trade_category_middle"] = safe_col(trade_df, "상품분류(중)").map(clean_text)
trade_df["trade_category_small"] = safe_col(trade_df, "상품분류(소)").map(clean_text)

trade_df["trade_category_path"] = (
    trade_df["trade_category_large"] + " > " +
    trade_df["trade_category_middle"] + " > " +
    trade_df["trade_category_small"]
).map(clean_text)

trade_df["bulk_price"] = safe_col(trade_df, "대량가격(원)").map(to_number)
trade_df["middle_price"] = safe_col(trade_df, "중간가격(원)").map(to_number)
trade_df["small_price"] = safe_col(trade_df, "소량가격(원)").map(to_number)
trade_df["trade_moq"] = safe_col(trade_df, "최소구매수량").map(to_number)
trade_df["print_method"] = safe_col(trade_df, "인쇄 방법").map(clean_text)

trade_df["trade_search_text"] = (
    trade_df["buyer_type"] + " " +
    trade_df["buyer_name"] + " " +
    trade_df["trade_product_name"] + " " +
    trade_df["trade_category_path"] + " " +
    trade_df["print_method"]
).map(clean_text)

trade_df = trade_df[trade_df["trade_product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 거래 수:", len(trade_df))
display(trade_df[[
    "buyer_type", "buyer_name", "trade_product_name",
    "trade_category_path", "bulk_price", "middle_price", "small_price",
    "trade_moq", "print_method"
]].head(10))


정규화 거래 수: 100


,buyer_type,buyer_name,trade_product_name,trade_category_path,bulk_price,middle_price,small_price,trade_moq,print_method
0,기업/회사 > 제약/의료기기/바이오 > 의료기기/의료용품 > 의료기기/의료용품,파라곤CRT,쏘쏘 원터치 스텐텀블러 500ml 1P,일상용품 > 주방용품 > 저장용기,4500.0,4578.0,5374.0,20.0,칼라전사필름인쇄
1,기업/회사 > 일반기업/회사 > 화장품/패션/주얼리 > 화장품,Changefit,귀요미 캐릭터 스마트링 1P,디지털/가전 > 휴대폰액세서리 > 기타휴대폰액세서리,850.0,873.0,989.0,50.0,칼라인쇄
2,기업/회사 > 일반기업/회사 > 외국계회사/글로벌기업 > 외국계회사/글로벌기업,CONMAD,에스모도 미니 핸디형 선풍기 159,디지털/가전 > 건강/냉난방가전 > 선풍기,7240.0,7556.0,8458.0,10.0,칼라인쇄
3,학교/교육기관 > 대학교/대학원 > 취업/창업지원/일자리센터 > 취업/창업지원/일자리센터,성결대학교 대학일자리센터,(완판) [카카오프렌즈] 코마사 세면타월 160g 1P,일상용품 > 욕실/청소용품 > 타월,NaN,NaN,NaN,NaN,-
4,기업/회사 > 전문직 > 세무사/회계법인 > 세무사/회계법인,신원세무회계,2025년 포인트세법 탁상달력 캘린더 카렌다 250*190mm 1P,교육/문화용품 > 문구/사무용품 > 종이류,990.0,1135.0,1523.0,100.0,"금,은,청,먹박인쇄"
5,기업/회사 > 일반기업/회사 > 외국계회사/글로벌기업 > 외국계회사/글로벌기업,MYNAVI KOREA CORPORATION,(인쇄 무료 이벤트) [마이브렐라] C형 수동 거꾸로 장우산 1P,패션잡화 > 패션소품 > 우산,8440.0,8730.0,9894.0,10.0,실크인쇄
6,문화시설/스포츠관련 > 골프관련 > 국립공원/수목원 > 국립공원/수목원,JIRISAN DULLEBOGO,[뮤스트] 팝스 USB 2.0 메모리 (4GB~128GB) 1P,디지털/가전 > 저장장치 > 스토리지,3800.0,3948.0,4278.0,20.0,레이저인쇄
7,기업/회사 > 일반기업/회사 > 식품회사/건강식품 > 식품회사/건강식품,JAWS,국내산쿨마스크 일반형 패션마스크 1P,일상용품 > 생활용품 > 생활잡화류,830.0,NaN,NaN,NaN,칼라전사인쇄
8,관공서/공공기관 > 보건소/정신건강/보건복지관련 > 보건소 > 보건소,부안군보건소,[울트라웨이브] 휴대용 마스크 살균건조기 MS-01 1P,디지털/가전 > 생활가전 > 건조기/탈수기,16000.0,16296.0,18139.0,5.0,칼라전사필름인쇄
9,학교/교육기관 > 중고등학교 > 고등학교 > 고등학교,계명고등학교,전도 종이투명칼라 지갑티슈 (10조 20매) 1P,일상용품 > 위생용품 > 화장지,144.0,147.0,196.0,2000.0,-


## 7. 검색 인덱스 만들기

v0에서는 벡터DB 없이 TF-IDF 문자 n-gram 검색을 사용합니다.

한국어는 띄어쓰기나 형태소 분석 문제가 있어서, 초반 baseline으로 문자 n-gram이 안정적입니다.


In [8]:
# =========================
# 상품/거래 검색 인덱스
# =========================

product_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1,
)

product_matrix = product_vectorizer.fit_transform(product_df["search_text"].fillna("").tolist())

trade_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1,
)

trade_matrix = trade_vectorizer.fit_transform(trade_df["trade_search_text"].fillna("").tolist())

print("상품 matrix:", product_matrix.shape)
print("거래 matrix:", trade_matrix.shape)


상품 matrix: (100, 16922)
거래 matrix: (100, 11023)


## 8. 사용자 질문에서 조건 추출

v0에서는 정규식으로 예산/수량만 간단히 추출합니다.

예:

```text
3천원 이하 → budget = 3000
5만원 → budget = 50000
500개 → quantity = 500
```


In [9]:
# =========================
# 조건 추출
# =========================

def extract_conditions(query: str) -> dict:
    q = str(query)

    budget = None

    # 예: 3000원, 3,000원
    m = re.search(r"(\d+(?:,\d+)*)\s*원\s*(?:이하|미만|안쪽|대로)?", q)
    if m:
        budget = int(m.group(1).replace(",", ""))

    # 예: 3천원, 3천 원, 3천원대
    if budget is None:
        m = re.search(r"(\d+)\s*천\s*원(?:대)?", q)
        if m:
            budget = int(m.group(1)) * 1000

    # 예: 5만원, 5만 원
    if budget is None:
        m = re.search(r"(\d+)\s*만\s*원", q)
        if m:
            budget = int(m.group(1)) * 10000

    quantity = None

    # 예: 500개, 1,000개, 500EA
    m = re.search(r"(\d+(?:,\d+)*)\s*(?:개|ea|EA|pcs|PCS)", q)
    if m:
        quantity = int(m.group(1).replace(",", ""))

    return {
        "raw_query": q,
        "budget": budget,
        "quantity": quantity,
    }


# 테스트
for q in [
    "병원 개원 답례품으로 3천원 이하 상품 추천해줘",
    "대학교 행사에서 500개 정도 나눠줄 2,000원대 사은품",
    "회사 창립기념품으로 5만원 이하 고급 상품 추천",
]:
    print(q, "=>", extract_conditions(q))


병원 개원 답례품으로 3천원 이하 상품 추천해줘 => {'raw_query': '병원 개원 답례품으로 3천원 이하 상품 추천해줘', 'budget': 3000, 'quantity': None}
대학교 행사에서 500개 정도 나눠줄 2,000원대 사은품 => {'raw_query': '대학교 행사에서 500개 정도 나눠줄 2,000원대 사은품', 'budget': 2000, 'quantity': 500}
회사 창립기념품으로 5만원 이하 고급 상품 추천 => {'raw_query': '회사 창립기념품으로 5만원 이하 고급 상품 추천', 'budget': 50000, 'quantity': None}


## 9. 거래데이터에서 과거 구매 힌트 찾기

거래데이터는 상품명 매칭용이 아니라,  
**비슷한 업종/행사/구매처에서 어떤 상품군이 자주 등장했는지** 확인하는 용도로 사용합니다.


In [10]:
# =========================
# 거래 힌트 검색
# =========================

def get_trade_hints(query: str, top_k: int = 10):
    if len(trade_df) == 0:
        return trade_df.head(0), []

    query_vec = trade_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, trade_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    hints = trade_df.iloc[top_indices].copy()
    hints.insert(0, "trade_score", scores[top_indices])

    # 검색된 거래 사례에서 자주 등장한 상품분류 힌트 추출
    category_candidates = []

    for col in ["trade_category_middle", "trade_category_small"]:
        if col in hints.columns:
            category_candidates.extend([
                clean_text(x)
                for x in hints[col].tolist()
                if clean_text(x)
            ])

    if category_candidates:
        category_hints = (
            pd.Series(category_candidates)
            .value_counts()
            .head(5)
            .index
            .tolist()
        )
    else:
        category_hints = []

    return hints, category_hints


# 테스트
trade_hints, category_hints = get_trade_hints("병원 개원 답례품으로 3천원 이하 상품 추천해줘", top_k=10)

print("거래데이터 기반 카테고리 힌트:", category_hints)
display(trade_hints[[
    "trade_score", "buyer_type", "buyer_name", "trade_product_name",
    "trade_category_path", "print_method"
]].head(10))


거래데이터 기반 카테고리 힌트: ['문구/사무용품', '생활용품', '필기용품', '생활잡화류', '생활가전']


,trade_score,buyer_type,buyer_name,trade_product_name,trade_category_path,print_method
26,0.205879,기업/회사 > 병원/의료기관 > 한의원/한방병원 > 한의원/한방병원,한가은한방병원,독일심 몽블랑 볼펜 (금장/은장) 1.0mm 1P,교육/문화용품 > 문구/사무용품 > 필기용품,레이저인쇄
72,0.096911,자영업 > 학원 > 수학전문학원 > 수학전문학원,위즈클래스 수학학원,컬러 배색 에코백 (36*40cm) 1P,일상용품 > 생활용품 > 생활잡화류,실크인쇄
82,0.067626,자영업 > 학원 > 영어학원/어학원 > 영어학원/어학원,성북 리드앤톡 영어도서관,메이트 지우개 (샤프식),교육/문화용품 > 문구/사무용품 > 필기용품,실크인쇄
83,0.065799,기업/회사 > 병원/의료기관 > 안과 > 안과,아이리움안과,PS 방수 보온보냉가방 (대/중/소) 1P,일상용품 > 생활용품 > 생활잡화류,실크인쇄
93,0.055141,학교/교육기관 > 유치원/어린이집 > 유치원 > 유치원,은솔유치원,[유비세이프] 휴대용 칫솔살균기 소독기 TS-501 LAMP (건전지전원+케이블상시연결전원),디지털/가전 > 생활가전 > 구강청정기,실크인쇄
92,0.049882,기업/회사 > 병원/의료기관 > 성형외과 > 성형외과,여수 조 성형외과,행잉세면 워시백 세면파우치 화장품파우치 여행 1P,일상용품 > 생활용품 > 생활잡화류,칼라전사인쇄
41,0.048899,학교/교육기관 > 대학교/대학원 > 산학협력관련 > 대학혁신지원사업,HS혁신지원 사업단,[PGA] 3단로고바이어스 완전자동 우산+260g 코마사 타월 세트,"패션잡화,일상용품 > 패션소품,욕실/청소용품 > 우산,타월",실크인쇄
74,0.048252,"관공서/공공기관 > 정부기관,관공서 > 연구원/연구소/연구센터/기술원 > 연구원/연구소/연구센터/기술원",국립과학수사연구원,알약 소화기 (가정 사무실 차량용 간이소화용구),가구/인테리어 > DIY용품 > 안전/보호용품,실크인쇄
6,0.036862,문화시설/스포츠관련 > 골프관련 > 국립공원/수목원 > 국립공원/수목원,JIRISAN DULLEBOGO,[뮤스트] 팝스 USB 2.0 메모리 (4GB~128GB) 1P,디지털/가전 > 저장장치 > 스토리지,레이저인쇄
53,0.036276,학교/교육기관 > 대학교/대학원 > 학과/학부 > 학과/학부,성결대학교 관광개발학과,[제스] 초저점도 3색터치볼펜 0.8mm 1P,교육/문화용품 > 문구/사무용품 > 필기용품,실크인쇄


## 10. 상품 추천 함수

점수 구조:

```text
최종점수 =
상품 검색 유사도
+ 거래데이터 카테고리 힌트 보너스
+ 예산 조건 보너스
+ 수량 조건 보너스/패널티
```

주의:

- v0에서는 가격 조건을 완전히 엄격하게 적용하지 않습니다.
- 샘플 데이터 100개라서 후보가 너무 적을 수 있기 때문입니다.
- 원본 데이터에서는 가격 필터를 더 강하게 적용할 수 있습니다.


In [11]:
# =========================
# 상품 추천 함수
# =========================

def recommend_products(
    query: str,
    top_k: int = 5,
    trade_top_k: int = 10,
    min_candidates: int = 10,
):
    conditions = extract_conditions(query)

    candidates = product_df.copy()
    filter_notes = []

    # 1) 예산 조건
    # 샘플 데이터에서는 후보가 너무 적을 수 있어, 후보 수가 충분할 때만 엄격 필터 적용
    if conditions["budget"] is not None and candidates["price"].notna().any():
        budget = conditions["budget"]

        strict_candidates = candidates[
            candidates["price"].notna() &
            (candidates["price"] <= budget)
        ]

        if len(strict_candidates) >= min_candidates:
            candidates = strict_candidates
            filter_notes.append(f"판매가 {budget:,}원 이하 필터 적용")
        else:
            loose_candidates = candidates[
                candidates["price"].notna() &
                (candidates["price"] <= budget * 1.2)
            ]

            if len(loose_candidates) >= min_candidates:
                candidates = loose_candidates
                filter_notes.append(f"후보 부족으로 판매가 {int(budget * 1.2):,}원 이하 완화 적용")
            else:
                filter_notes.append("예산 조건 후보가 적어 가격은 점수에만 반영")

    # 2) 거래데이터에서 카테고리 힌트 추출
    trade_hints, category_hints = get_trade_hints(query, top_k=trade_top_k)

    # 3) 상품 검색
    # 거래 힌트를 쿼리에 살짝 추가해 상품 검색을 보강
    augmented_query = query + " " + " ".join(category_hints)

    query_vec = product_vectorizer.transform([augmented_query])
    product_scores = cosine_similarity(query_vec, product_matrix).flatten()

    result_df = candidates.copy()

    # candidates의 index는 product_df 기준 index이므로 그대로 product_scores에서 조회
    result_df["base_score"] = result_df.index.map(lambda i: float(product_scores[i]))

    # 4) 거래 힌트 보너스
    def calc_trade_hint_bonus(row):
        text = combine_text(
            row.get("product_name", ""),
            row.get("category_path", ""),
            row.get("keywords", ""),
        )

        if any(hint and hint in text for hint in category_hints):
            return 0.08

        return 0.0

    result_df["trade_hint_bonus"] = result_df.apply(calc_trade_hint_bonus, axis=1)

    # 5) 예산 보너스
    if conditions["budget"] is not None:
        budget = conditions["budget"]

        def calc_budget_bonus(price):
            if pd.isna(price):
                return 0.0
            if price <= budget:
                return 0.05
            if price <= budget * 1.2:
                return 0.02
            return -0.03

        result_df["budget_bonus"] = result_df["price"].apply(calc_budget_bonus)
    else:
        result_df["budget_bonus"] = 0.0

    # 6) 수량 보너스
    if conditions["quantity"] is not None:
        qty = conditions["quantity"]

        result_df["qty_ok"] = result_df["moq"].apply(
            lambda x: True if pd.isna(x) else x <= qty
        )

        result_df["quantity_bonus"] = result_df["qty_ok"].apply(
            lambda ok: 0.03 if ok else -0.03
        )
    else:
        result_df["qty_ok"] = ""
        result_df["quantity_bonus"] = 0.0

    # 7) 최종 점수
    result_df["final_score"] = (
        result_df["base_score"] +
        result_df["trade_hint_bonus"] +
        result_df["budget_bonus"] +
        result_df["quantity_bonus"]
    )

    # 추천 이유 생성
    def make_reason(row):
        reasons = []

        if conditions["budget"] is not None and pd.notna(row["price"]):
            if row["price"] <= conditions["budget"]:
                reasons.append(f"예산 {conditions['budget']:,}원 이하 조건에 맞음")
            elif row["price"] <= conditions["budget"] * 1.2:
                reasons.append("예산보다 약간 높지만 후보 부족으로 검토 가능")
            else:
                reasons.append("예산 초과 가능성 있음")

        if conditions["quantity"] is not None and row["qty_ok"] != "":
            if row["qty_ok"]:
                reasons.append("요청 수량 기준 최소구매수량 충족 가능")
            else:
                reasons.append("최소구매수량 확인 필요")

        if row["trade_hint_bonus"] > 0:
            reasons.append("과거 거래데이터의 유사 상품군과 관련 있음")

        if row["base_score"] > 0.2:
            reasons.append("요청 문구와 상품명/카테고리/키워드 유사도 높음")
        elif row["base_score"] > 0.1:
            reasons.append("요청 문구와 일부 관련 있음")

        if not reasons:
            reasons.append("상품명/카테고리 기준 후보로 검토 가능")

        return " / ".join(reasons)

    result_df["recommend_reason"] = result_df.apply(make_reason, axis=1)

    output_cols = [
        "product_id",
        "product_name",
        "price",
        "moq",
        "category_path",
        "keywords",
        "base_score",
        "trade_hint_bonus",
        "budget_bonus",
        "quantity_bonus",
        "final_score",
        "qty_ok",
        "recommend_reason",
        "image",
    ]

    recommendations = (
        result_df
        .sort_values("final_score", ascending=False)
        .head(top_k)
        [output_cols]
        .reset_index(drop=True)
    )

    return {
        "query": query,
        "conditions": conditions,
        "filter_notes": filter_notes,
        "category_hints": category_hints,
        "trade_hints": trade_hints,
        "recommendations": recommendations,
    }


## 11. 추천 테스트

아래 질문을 바꿔가며 테스트합니다.


In [12]:
# =========================
# 추천 테스트
# =========================

test_query = "병원 개원 답례품으로 3천원 이하 상품 추천해줘"

result = recommend_products(test_query, top_k=5)

print("[질문]")
print(result["query"])

print("\n[추출 조건]")
print(result["conditions"])

print("\n[필터 메모]")
for note in result["filter_notes"]:
    print("-", note)

print("\n[거래데이터 기반 카테고리 힌트]")
print(result["category_hints"])

print("\n[추천 상품]")
display(result["recommendations"])

print("\n[참고 거래 사례]")
display(result["trade_hints"][[
    "trade_score",
    "buyer_type",
    "buyer_name",
    "trade_product_name",
    "trade_category_path",
    "print_method",
]].head(5))


[질문]
병원 개원 답례품으로 3천원 이하 상품 추천해줘

[추출 조건]
{'raw_query': '병원 개원 답례품으로 3천원 이하 상품 추천해줘', 'budget': 3000, 'quantity': None}

[필터 메모]
- 판매가 3,000원 이하 필터 적용

[거래데이터 기반 카테고리 힌트]
['문구/사무용품', '생활용품', '필기용품', '생활잡화류', '생활가전']

[추천 상품]


,product_id,product_name,price,moq,category_path,keywords,base_score,trade_hint_bonus,budget_bonus,quantity_bonus,final_score,qty_ok,recommend_reason,image
0,101874,[송월타올] 메리스타 1매 기념수건 답례품,2923.0,10000.0,가정/생활용품 > 수건(타월) > 일반타월,#송월타올 #송월타월 #송월수건 #크리스마스선물 #크리스마스타올 #자수타올 #크리스마스타올세트 #세면타월 #개업수건 #개업타올 #답례품타올 #행사용타올 #기념타올 #타올제작 #낭만타올 #개업선물 #기업행사 #기념수건 #기념타월 #돌답례품,0.045859,0.08,0.05,0.0,0.175859,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음",판촉배너_24.jpg
1,56036,에코밴드세트 소D (95*66*16mm),1187.0,30000.0,가정/생활용품 > 응급 / 안전용품 > 밴드세트,#밴드세트,0.043876,0.08,0.05,0.0,0.173876,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음",02/81610002.jpg
2,81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품,"판촉물,사은품,기념품,답례품,선물용품,선물,홍보물,홍보용품,장갑,고무장갑,주방장갑,다용도장갑,위생장갑,멀티장갑,기모장갑,기모고무장갑,라텍스장갑",0.039032,0.08,0.05,0.0,0.169032,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음",231103G.jpg
3,68823,미니손톱깎이(브라운패턴),2034.0,10000.0,가정/생활용품 > 손톱깎이세트 > 손톱깎이,#손톱깍이세트 #손톱깍이,0.028375,0.08,0.05,0.0,0.158375,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음",2020072554927530.jpg
4,84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월,#타올#수건#미용타올#미용실타올#극세사수건#극세사타올,0.023794,0.08,0.05,0.0,0.153794,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음",뷰티미용타올(메인).jpg



[참고 거래 사례]


,trade_score,buyer_type,buyer_name,trade_product_name,trade_category_path,print_method
26,0.205879,기업/회사 > 병원/의료기관 > 한의원/한방병원 > 한의원/한방병원,한가은한방병원,독일심 몽블랑 볼펜 (금장/은장) 1.0mm 1P,교육/문화용품 > 문구/사무용품 > 필기용품,레이저인쇄
72,0.096911,자영업 > 학원 > 수학전문학원 > 수학전문학원,위즈클래스 수학학원,컬러 배색 에코백 (36*40cm) 1P,일상용품 > 생활용품 > 생활잡화류,실크인쇄
82,0.067626,자영업 > 학원 > 영어학원/어학원 > 영어학원/어학원,성북 리드앤톡 영어도서관,메이트 지우개 (샤프식),교육/문화용품 > 문구/사무용품 > 필기용품,실크인쇄
83,0.065799,기업/회사 > 병원/의료기관 > 안과 > 안과,아이리움안과,PS 방수 보온보냉가방 (대/중/소) 1P,일상용품 > 생활용품 > 생활잡화류,실크인쇄
93,0.055141,학교/교육기관 > 유치원/어린이집 > 유치원 > 유치원,은솔유치원,[유비세이프] 휴대용 칫솔살균기 소독기 TS-501 LAMP (건전지전원+케이블상시연결전원),디지털/가전 > 생활가전 > 구강청정기,실크인쇄


## 12. 여러 질문 일괄 테스트

추천 결과를 엑셀로 저장합니다.


In [13]:
# =========================
# 여러 질문 테스트
# =========================

test_queries = [
    "병원 개원 답례품으로 3천원 이하 상품 추천해줘",
    "대학교 행사에서 나눠줄 2천원대 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 저렴한 홍보물 추천해줘",
    "여름 행사에 어울리는 판촉물 추천해줘",
    "겨울철 고객 사은품 추천해줘",
    "500개 정도 주문할 수 있는 물티슈 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
    "예산 5천원 이하로 고급스러운 기념품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
]

all_rows = []

for query in test_queries:
    result = recommend_products(query, top_k=5)

    for rank, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "base_score": row["base_score"],
            "trade_hint_bonus": row["trade_hint_bonus"],
            "budget_bonus": row["budget_bonus"],
            "quantity_bonus": row["quantity_bonus"],
            "final_score": row["final_score"],
            "qty_ok": row["qty_ok"],
            "recommend_reason": row["recommend_reason"],
            "category_hints": ", ".join(result["category_hints"]),
            "filter_notes": " / ".join(result["filter_notes"]),
        })

batch_result_df = pd.DataFrame(all_rows)

BATCH_RESULT_PATH = OUTPUT_DIR / "giftco_reco_v0_tfidf_batchtest.xlsx"
batch_result_df.to_excel(BATCH_RESULT_PATH, index=False)

print("추천 테스트 결과 저장:", BATCH_RESULT_PATH)
display(batch_result_df.head(20))


추천 테스트 결과 저장: output\giftco_reco_v0_tfidf_batchtest.xlsx


,query,rank,product_id,product_name,price,moq,category_path,base_score,trade_hint_bonus,budget_bonus,quantity_bonus,final_score,qty_ok,recommend_reason,category_hints,filter_notes
0,병원 개원 답례품으로 3천원 이하 상품 추천해줘,1,101874,[송월타올] 메리스타 1매 기념수건 답례품,2923.0,10000.0,가정/생활용품 > 수건(타월) > 일반타월,0.045859,0.08,0.05,0.0,0.175859,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 필기용품, 생활잡화류, 생활가전","판매가 3,000원 이하 필터 적용"
1,병원 개원 답례품으로 3천원 이하 상품 추천해줘,2,56036,에코밴드세트 소D (95*66*16mm),1187.0,30000.0,가정/생활용품 > 응급 / 안전용품 > 밴드세트,0.043876,0.08,0.05,0.0,0.173876,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 필기용품, 생활잡화류, 생활가전","판매가 3,000원 이하 필터 적용"
2,병원 개원 답례품으로 3천원 이하 상품 추천해줘,3,81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품,0.039032,0.08,0.05,0.0,0.169032,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 필기용품, 생활잡화류, 생활가전","판매가 3,000원 이하 필터 적용"
3,병원 개원 답례품으로 3천원 이하 상품 추천해줘,4,68823,미니손톱깎이(브라운패턴),2034.0,10000.0,가정/생활용품 > 손톱깎이세트 > 손톱깎이,0.028375,0.08,0.05,0.0,0.158375,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 필기용품, 생활잡화류, 생활가전","판매가 3,000원 이하 필터 적용"
4,병원 개원 답례품으로 3천원 이하 상품 추천해줘,5,84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월,0.023794,0.08,0.05,0.0,0.153794,,"예산 3,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 필기용품, 생활잡화류, 생활가전","판매가 3,000원 이하 필터 적용"
5,대학교 행사에서 나눠줄 2천원대 사은품 추천해줘,1,81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품,0.052021,0.08,0.05,0.0,0.182021,,"예산 2,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 생활잡화류, 필기용품, 패션의류","판매가 2,000원 이하 필터 적용"
6,대학교 행사에서 나눠줄 2천원대 사은품 추천해줘,2,56036,에코밴드세트 소D (95*66*16mm),1187.0,30000.0,가정/생활용품 > 응급 / 안전용품 > 밴드세트,0.051179,0.08,0.05,0.0,0.181179,,"예산 2,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 생활잡화류, 필기용품, 패션의류","판매가 2,000원 이하 필터 적용"
7,대학교 행사에서 나눠줄 2천원대 사은품 추천해줘,3,41118,신형왕부채(R형) 280x230mm,418.0,50000.0,가정/생활용품 > 부채 > 홍보용부채,0.029504,0.08,0.05,0.0,0.159504,,"예산 2,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 생활잡화류, 필기용품, 패션의류","판매가 2,000원 이하 필터 적용"
8,대학교 행사에서 나눠줄 2천원대 사은품 추천해줘,4,84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월,0.027240,0.08,0.05,0.0,0.157240,,"예산 2,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 생활잡화류, 필기용품, 패션의류","판매가 2,000원 이하 필터 적용"
9,대학교 행사에서 나눠줄 2천원대 사은품 추천해줘,5,56194,민무늬(무지) 스카프 손수건 두건,1017.0,5000.0,가정/생활용품 > 수건(타월) > 손수건/스카프,0.026661,0.08,0.05,0.0,0.156661,,"예산 2,000원 이하 조건에 맞음 / 과거 거래데이터의 유사 상품군과 관련 있음","문구/사무용품, 생활용품, 생활잡화류, 필기용품, 패션의류","판매가 2,000원 이하 필터 적용"


## 13. Ollama로 추천 설명 생성하기

선택 사항입니다.

사전 준비:

```bash
pip install ollama
ollama pull gemma3
```

또는 한국어가 더 나은 모델을 쓰고 싶으면:

```bash
ollama pull qwen3
```

이 셀은 추천 상품 리스트를 바탕으로 고객에게 보여줄 자연어 답변을 생성합니다.


In [14]:
# =========================
# Ollama 답변 생성
# =========================

def build_recommendation_context(result: dict) -> str:
    recs = result["recommendations"]
    hints = result["trade_hints"].head(5)

    lines = []

    lines.append("[추출 조건]")
    lines.append(str(result["conditions"]))
    lines.append("")

    lines.append("[추천 상품 후보]")
    for i, (_, row) in enumerate(recs.iterrows(), start=1):
        lines.append(f"{i}. {row['product_name']}")
        lines.append(f"   - 가격: {row['price']}")
        lines.append(f"   - 최소구매수량: {row['moq']}")
        lines.append(f"   - 카테고리: {row['category_path']}")
        lines.append(f"   - 추천근거: {row['recommend_reason']}")
        lines.append("")

    lines.append("[참고 거래 사례]")
    for i, (_, row) in enumerate(hints.iterrows(), start=1):
        lines.append(f"{i}. {row['trade_product_name']}")
        lines.append(f"   - 구매처유형: {row['buyer_type']}")
        lines.append(f"   - 상품분류: {row['trade_category_path']}")
        lines.append("")

    return "\n".join(lines)


def generate_recommendation_answer_with_ollama(
    query: str,
    model: str = "gemma3",
    top_k: int = 5,
):
    try:
        import ollama
    except ModuleNotFoundError:
        raise ModuleNotFoundError(
            "ollama 패키지가 설치되어 있지 않습니다. "
            "현재 가상환경에서 `python -m pip install ollama`를 실행하세요."
        )

    result = recommend_products(query, top_k=top_k)
    context = build_recommendation_context(result)

    system_prompt = """
너는 기프트코 판촉물 상품 추천 어시스턴트다.

규칙:
- 제공된 추천 후보와 거래 사례만 근거로 답변한다.
- 가격, 최소구매수량, 납기 등 데이터에 없는 내용은 단정하지 않는다.
- 고객에게는 Top 3~5개 상품을 간단히 추천한다.
- 각 상품마다 추천 이유를 붙인다.
- 조건이 부족하면 추가 확인이 필요한 항목을 마지막에 적는다.
- 자연스러운 한국어로 답변한다.
"""

    user_prompt = f"""
[사용자 요청]
{query}

[추천 근거 데이터]
{context}

위 근거만 바탕으로 고객에게 상품 추천 답변을 작성해줘.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()},
        ],
        options={
            "temperature": 0.2,
            "num_ctx": 4096,
        },
        stream=False,
    )

    return {
        "query": query,
        "answer": response["message"]["content"],
        "result": result,
    }

In [ ]:
ollama_result = generate_recommendation_answer_with_ollama(
    "병원 개원 답례품으로 3천원 이하 상품 추천해줘",
    model="gemma3:4b",
    top_k=5,
)
print(ollama_result["answer"])
display(ollama_result["result"]["recommendations"])

# 다음 단계

이 v0를 실행한 뒤 확인할 것:

1. 가격 필터가 과하게 작동하지 않는가?
2. 거래데이터 힌트가 상품 추천을 오히려 이상하게 만들지는 않는가?
3. 상품명이 아닌 카테고리 기준으로도 추천이 잘 되는가?
4. 상품데이터의 `검색키워드`, `카테고리`, `상품내용` 품질이 충분한가?
5. 추천 결과에 실제로 판매 가능한 상품이 나오는가?

## 다음 버전

다음: v1에서 질문을 구조화하고, 거래데이터를 임베딩 기반으로 검색합니다. 전체 버전 로드맵과 배경은 `README.md`를 참고하세요.
